In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# PART B - Implementation

In [2]:
# The dataset columns are:
# 0: polarity (0 = negative, 2 = neutral, 4 = positive)
# 1: id
# 2: date
# 3: query
# 4: user
# 5: text

DATASET_COLUMNS = ['polarity', 'id', 'date', 'query', 'user', 'text']
DATASET_ENCODING = "ISO-8859-1"

data_path = './dataset.csv'

# The dataset columns are:
# 0: polarity (0 = negative, 2 = neutral, 4 = positive)
# 1: id
# 2: date
# 3: query
# 4: user
# 5: text

DATASET_COLUMNS = ['polarity', 'id', 'date', 'query', 'user', 'text']
DATASET_ENCODING = "ISO-8859-1"

try:
    df = pd.read_csv(data_path, encoding=DATASET_ENCODING, names=DATASET_COLUMNS)
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print(f"Error: The file '{data_path}' was not found. Please ensure the dataset is correctly linked.")
    # Exit the script or handle the error gracefully if needed
    exit()
print(df['polarity'].value_counts())
df_stratified = (
    df.groupby('polarity', group_keys=False)
      .apply(lambda x: x.sample(int(50000 * len(x) / len(df)), random_state=42))
)

print(df_stratified['polarity'].value_counts())
print("Final shape:", df_stratified.shape)

df=df_stratified
df['polarity'].value_counts()


Dataset loaded successfully.
polarity
0    800000
4    800000
Name: count, dtype: int64
polarity
0    25000
4    25000
Name: count, dtype: int64
Final shape: (50000, 6)


C:\Users\affan\AppData\Local\Temp\ipykernel_21432\2683475787.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(int(50000 * len(x) / len(df)), random_state=42))


polarity
0    25000
4    25000
Name: count, dtype: int64

# RNN - Recurrent Neural Network with LSTM

In [3]:
# Part B: Implementation of Sentiment Analysis on Twitter Data

# Step 1: Install and Import Necessary Libraries
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.metrics import Precision, Recall

# Download NLTK data
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Step 2: Data Acquisition and Loading
data_path = './dataset.csv'

# The dataset columns are:
# 0: polarity (0 = negative, 2 = neutral, 4 = positive)
# 1: id
# 2: date
# 3: query
# 4: user
# 5: text

DATASET_COLUMNS = ['polarity', 'id', 'date', 'query', 'user', 'text']
DATASET_ENCODING = "ISO-8859-1"

try:
    df = pd.read_csv(data_path, encoding=DATASET_ENCODING, names=DATASET_COLUMNS)
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print(f"Error: The file '{data_path}' was not found. Please ensure the dataset is correctly linked.")
    exit()

print("Original dataset polarity distribution:")
print(df['polarity'].value_counts())

# Create stratified sample (50K from each class)
df_stratified = (
    df.groupby('polarity', group_keys=False)
      .apply(lambda x: x.sample(int(50000 * len(x) / len(df)), random_state=42))
)

print("\nStratified dataset polarity distribution:")
print(df_stratified['polarity'].value_counts())
print("Final shape:", df_stratified.shape)

df = df_stratified

# Step 3: Data Preprocessing
print("\nStarting data preprocessing...")

# Filter only negative (0) and positive (4) tweets, remove neutral (2)
df = df[df['polarity'].isin([0, 4])]
print("After filtering neutral tweets:")
print(df['polarity'].value_counts())

# Convert polarity: 4 → 1 (positive), 0 → 0 (negative)
df['polarity'] = df['polarity'].replace(4, 1)
print("\nAfter converting labels (4→1):")
print(df['polarity'].value_counts())

# Improved text preprocessing
def clean_text(text):
    text = str(text).lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove user mentions and hashtags (but keep the text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    # Remove special characters but keep basic punctuation for sentiment
    text = re.sub(r'[^a-zA-Z\s!?]', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Keep some stopwords that carry sentiment
def get_sentiment_stopwords():
    base_stopwords = set(stopwords.words('english'))
    # Keep words that are important for sentiment
    sentiment_words = {'not', 'no', 'never', 'none', 'nothing', 'nobody', 
                      'nowhere', 'neither', 'nor', 'cannot', 'couldn', 
                      "couldn't", 'didn', "didn't", 'doesn', "doesn't", 
                      'hadn', "hadn't", 'hasn', "hasn't", 'haven', "haven't", 
                      'isn', "isn't", 'mightn', "mightn't", 'mustn', "mustn't", 
                      'needn', "needn't", 'shan', "shan't", 'shouldn', "shouldn't", 
                      'wasn', "wasn't", 'weren', "weren't", 'won', "won't", 
                      'wouldn', "wouldn't", 'very', 'too', 'extremely', 'quite',
                      'but', 'however', 'although', 'though', 'despite'}
    return base_stopwords - sentiment_words

stop_words = get_sentiment_stopwords()
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    words = text.split()
    # Keep negation words and intensifiers
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(words)

print("Cleaning and preprocessing text...")
df['text'] = df['text'].apply(clean_text)
df['text'] = df['text'].apply(preprocess_text)

# Remove empty texts after preprocessing
df = df[df['text'].str.strip().astype(bool)]
print(f"After removing empty texts: {df.shape}")

# Step 4: Split the data into training and testing sets
X = df['text']
y = df['polarity']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining data: {len(X_train)} samples")
print(f"Testing data: {len(X_test)} samples")

# Diagnostic checks
print("\nClass distribution in training data:")
print(y_train.value_counts())

# Check class imbalance
positive_ratio = y_train.value_counts()[1] / len(y_train)
negative_ratio = y_train.value_counts()[0] / len(y_train)
print(f"Positive ratio: {positive_ratio:.3f}")
print(f"Negative ratio: {negative_ratio:.3f}")

if abs(positive_ratio - negative_ratio) > 0.1:
    print("⚠️  Class imbalance detected - applying class weights")

# Step 5: Tokenize and pad the sequences
MAX_WORDS = 50000
MAX_SEQUENCE_LENGTH = 100

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<unk>")
tokenizer.fit_on_texts(X_train)

X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_test_sequences = tokenizer.texts_to_sequences(X_test)

X_train_padded = pad_sequences(X_train_sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
X_test_padded = pad_sequences(X_test_sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')

print(f"\nVocabulary size: {len(tokenizer.word_index)}")
print(f"Padded sequences shape: {X_train_padded.shape}")

# Step 6: Build the Keras Model
embedding_dim = 128

model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=embedding_dim, 
              input_length=MAX_SEQUENCE_LENGTH, mask_zero=True),
    Dropout(0.3),
    LSTM(128, return_sequences=False, dropout=0.3, recurrent_dropout=0.3),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.4),
    Dense(1, activation='sigmoid')
])

# Calculate class weights
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))
print(f"Class weights: {class_weight_dict}")

model.compile(
    loss='binary_crossentropy', 
    optimizer='adam', 
    metrics=['accuracy', Precision(name="precision"), Recall(name="recall")]
)

# Build the model explicitly
dummy_input = np.zeros((1, MAX_SEQUENCE_LENGTH))
_ = model(dummy_input)

model.summary()

# Step 7: Train the Model
BATCH_SIZE = 64
EPOCHS = 15

# Enhanced callbacks
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    restore_best_weights=True,
    min_delta=0.001
)

checkpoint = ModelCheckpoint(
    "best_model.keras", 
    monitor='val_accuracy', 
    save_best_only=True,
    mode='max'
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.2, 
    patience=2, 
    min_lr=0.0001
)

print("\nStarting model training...")
history = model.fit(
    X_train_padded,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    class_weight=class_weight_dict,
    callbacks=[early_stop, checkpoint, reduce_lr],
    verbose=1
)

# Step 8: Evaluate the Model
print("\n" + "="*60)
print("COMPREHENSIVE MODEL EVALUATION")
print("="*60)

# Basic metrics
loss, accuracy, precision, recall = model.evaluate(X_test_padded, y_test)
print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")

# Detailed predictions
y_pred_proba = model.predict(X_test_padded)
y_pred = (y_pred_proba > 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Step 9: Enhanced Prediction Function
def predict_sentiment(text):
    cleaned_text = preprocess_text(clean_text(text))
    sequence = tokenizer.texts_to_sequences([cleaned_text])
    
    if not sequence[0]:  # If no words in vocabulary
        return "Neutral", 0.5, "No meaningful words found"
    
    padded_sequence = pad_sequences(sequence, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
    prediction = model.predict(padded_sequence, verbose=0)[0][0]
    
    # More nuanced sentiment classification
    if prediction >= 0.7:
        sentiment = 'Strongly Positive'
    elif prediction >= 0.6:
        sentiment = 'Positive'
    elif prediction >= 0.4:
        sentiment = 'Neutral'
    elif prediction >= 0.3:
        sentiment = 'Negative'
    else:
        sentiment = 'Strongly Negative'
    
    return sentiment, float(prediction), cleaned_text

# Step 10: Test with Various Tweets
print("\n" + "="*60)
print("TESTING WITH VARIOUS TWEETS")
print("="*60)

test_tweets = [
    # Positive tweets
    "The food at the restaurant was amazing! I had a great time.",
    "The food at the restaurant was bad! I had a worse time.",
]

for i, tweet in enumerate(test_tweets, 1):
    sentiment, confidence, cleaned = predict_sentiment(tweet)
    print(f"\nTest {i}:")
    print(f"Original: '{tweet}'")
    print(f"Cleaned: '{cleaned}'")
    print(f"Predicted: {sentiment} (Score: {confidence:.4f})")
    print("-" * 50)



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\affan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\affan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\affan\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Dataset loaded successfully.
Original dataset polarity distribution:
polarity
0    800000
4    800000
Name: count, dtype: int64


C:\Users\affan\AppData\Local\Temp\ipykernel_21432\3321590136.py:52: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(int(50000 * len(x) / len(df)), random_state=42))



Stratified dataset polarity distribution:
polarity
0    25000
4    25000
Name: count, dtype: int64
Final shape: (50000, 6)

Starting data preprocessing...
After filtering neutral tweets:
polarity
0    25000
4    25000
Name: count, dtype: int64

After converting labels (4→1):
polarity
0    25000
1    25000
Name: count, dtype: int64
Cleaning and preprocessing text...
After removing empty texts: (49851, 6)

Training data: 39880 samples
Testing data: 9971 samples

Class distribution in training data:
polarity
0    19949
1    19931
Name: count, dtype: int64
Positive ratio: 0.500
Negative ratio: 0.500

Vocabulary size: 32885
Padded sequences shape: (39880, 100)
Class weights: {0: 0.9995488495663943, 1: 1.0004515578746676}
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 100, 128)          6400000   
                                                       

# CNN

In [4]:
# Part B: Sentiment Analysis with CNN (Fixed)

import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# NLTK downloads
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Cleaning (keep ! ? because they carry sentiment)
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  # remove URLs
    text = re.sub(r'<.*?>+', '', text)                # HTML
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)        # mentions
    text = re.sub(r'#[A-Za-z0-9_]+', '', text)        # hashtags
    text = re.sub(r'[^a-z!? ]', '', text)             # keep a-z, !, ?
    return text

df['text'] = df['text'].apply(clean_text)

# Preprocess
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [lemmatizer.lemmatize(w) for w in words]
    return ' '.join(words)

df['text'] = df['text'].apply(preprocess_text)

# Split (stratify ensures class balance)
X = df['text']
y = df['polarity']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Tokenization & Padding
MAX_WORDS = 20000   # smaller vocab reduces overfitting
MAX_SEQUENCE_LENGTH = 100

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<unk>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_SEQUENCE_LENGTH)
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_SEQUENCE_LENGTH)

# Build CNN
embedding_dim = 128

model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=embedding_dim, input_length=MAX_SEQUENCE_LENGTH),
    Conv1D(128, 5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Build explicitly to show correct summary
model.build(input_shape=(None, MAX_SEQUENCE_LENGTH))
model.summary()

# Train
BATCH_SIZE = 128
EPOCHS = 10

es = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

print("\n🚀 Training CNN...")
history = model.fit(
    X_train_pad, y_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[es],
    verbose=1
)

# Evaluate
loss, accuracy = model.evaluate(X_test_pad, y_test)
print(f"\n📊 Test Data -> Loss: {loss:.4f}, Accuracy: {accuracy:.4f}")

# Predict
def predict_sentiment(text):
    cleaned = preprocess_text(clean_text(text))
    seq = tokenizer.texts_to_sequences([cleaned])
    pad = pad_sequences(seq, maxlen=MAX_SEQUENCE_LENGTH)
    pred = model.predict(pad)[0][0]
    return ('Positive' if pred > 0.5 else 'Negative', pred)

# Example
tweet = "The food at the restaurant was amazing! I had a great time."
sentiment, conf = predict_sentiment(tweet)
print(f"\n💬 '{tweet}' -> {sentiment} ({conf:.4f})")


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\affan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\affan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\affan\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 100, 128)          2560000   
                                                                 
 conv1d (Conv1D)             (None, 96, 128)           82048     
                                                                 
 global_max_pooling1d (Globa  (None, 128)              0         
 lMaxPooling1D)                                                  
                                                                 
 dense_3 (Dense)             (None, 64)                8256      
                                                                 
 dropout_3 (Dropout)         (None, 64)                0         
                                                                 
 dense_4 (Dense)             (None, 1)                 65        
                                                      

### Export Best Model as PKL
Save the best neural network model with tokenizer and preprocessing functions for use in Streamlit app


In [11]:
# Export Best Model as PKL for Streamlit
import pickle
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

class NeuralNetworkSentimentModel:
    """
    Wrapper class for the neural network model with tokenizer and preprocessing
    Rebuilds model from architecture parameters and loads weights from .h5 file
    """
    def __init__(self, weights_path, tokenizer, max_sequence_length, max_words, 
                 embedding_dim, clean_text_func, preprocess_text_func):
        self.weights_path = weights_path  # Path to .h5 weights file
        self.tokenizer = tokenizer
        self.max_sequence_length = max_sequence_length
        self.max_words = max_words
        self.embedding_dim = embedding_dim
        self.clean_text = clean_text_func
        self.preprocess_text = preprocess_text_func
        self._model = None  # Lazy loading - rebuild model when needed
    
    def _get_model(self):
        """Rebuild model from architecture and load weights"""
        if self._model is None:
            from tensorflow.keras.models import Sequential
            from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
            
            # Rebuild the LSTM model architecture (matching training code)
            self._model = Sequential([
                Embedding(input_dim=self.max_words, output_dim=self.embedding_dim, 
                          input_length=self.max_sequence_length, mask_zero=True),
                Dropout(0.3),
                LSTM(128, return_sequences=False, dropout=0.3, recurrent_dropout=0.3),
                Dense(64, activation='relu'),
                Dropout(0.5),
                Dense(32, activation='relu'),
                Dropout(0.4),
                Dense(1, activation='sigmoid')
            ])
            
            # Compile the model
            self._model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
            
            # Load weights
            import os
            if os.path.isabs(self.weights_path):
                weights_file = self.weights_path
            else:
                # Try relative to current directory first
                weights_file = os.path.abspath(self.weights_path)
                if not os.path.exists(weights_file):
                    # Try relative to current working directory (for notebooks)
                    weights_file = os.path.join(os.getcwd(), self.weights_path)
            
            if not os.path.exists(weights_file):
                # Try alternative paths
                alt_paths = [
                    "best_model_weights.h5",
                    "best_model_portable.keras",
                    "best_model.keras"
                ]
                for alt_path in alt_paths:
                    if os.path.exists(alt_path):
                        weights_file = alt_path
                        break
                else:
                    raise FileNotFoundError(
                        f"Weights file not found: {self.weights_path}\n"
                        f"Tried: {weights_file}\n"
                        f"Also tried: {alt_paths}\n"
                        f"Please ensure 'best_model_weights.h5' is in the same directory."
                    )
            
            # Load weights
            if weights_file.endswith('.h5'):
                self._model.load_weights(weights_file)
            else:
                # If it's a .keras file, try loading as full model
                from tensorflow.keras.models import load_model
                self._model.load_weights(weights_file)
                self._model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        return self._model
    
    def predict_sentiment(self, text):
        """Predict sentiment for a single text"""
        # Clean and preprocess
        cleaned = self.preprocess_text(self.clean_text(text))
        
        # Tokenize and pad
        seq = self.tokenizer.texts_to_sequences([cleaned])
        pad = pad_sequences(seq, maxlen=self.max_sequence_length)
        
        # Get model and predict
        model = self._get_model()
        pred = model.predict(pad, verbose=0)[0][0]
        
        # Return sentiment and confidence
        sentiment = 'Positive' if pred > 0.5 else 'Negative'
        confidence = pred if pred > 0.5 else 1 - pred
        
        return sentiment, float(pred), float(confidence)
    
    def predict(self, texts):
        """Predict sentiment for multiple texts (compatible with sklearn interface)"""
        if isinstance(texts, str):
            texts = [texts]
        
        results = []
        for text in texts:
            sentiment, prob, conf = self.predict_sentiment(text)
            # Return class label: 1 for Positive, 0 for Negative
            class_label = 1 if sentiment == 'Positive' else 0
            results.append(class_label)
        
        return np.array(results)

# Load model and save weights separately to avoid temp path issues
# This fixes issues with temporary paths from ModelCheckpoint
print("Loading model from best_model.keras...")
try:
    best_model = load_model("best_model.keras")
    print("✅ Model loaded successfully!")
    
    # Save weights separately as .h5 (portable format, no path issues)
    weights_path = "best_model_weights.h5"
    print(f"Saving model weights to {weights_path}...")
    best_model.save_weights(weights_path)
    print(f"✅ Model weights saved successfully to {weights_path}!")
    
    # Get embedding dimension from model (needed to rebuild)
    embedding_dim = best_model.layers[0].output_shape[-1]  # Embedding layer output dim
    print(f"✅ Model embedding dimension: {embedding_dim}")
    
    # Use weights file path
    model_file_path = "best_model_weights.h5"
    model_embedding_dim = embedding_dim
except Exception as e:
    print(f"⚠️ Warning: Could not process model: {e}")
    print("Trying to use original best_model.keras path...")
    model_file_path = "best_model.keras"
    model_embedding_dim = 128  # Default embedding dimension

# Create the wrapper with weights path and architecture parameters
# Using the tokenizer and constants from LSTM training section
nn_wrapper = NeuralNetworkSentimentModel(
    weights_path=model_file_path,  # Path to weights file (.h5)
    tokenizer=tokenizer,  # Tokenizer from LSTM section
    max_sequence_length=MAX_SEQUENCE_LENGTH,  # From LSTM section
    max_words=MAX_WORDS,  # From LSTM section
    embedding_dim=model_embedding_dim,  # Embedding dimension from model
    clean_text_func=clean_text,  # From LSTM section
    preprocess_text_func=preprocess_text  # From LSTM section
)

# Verify the model can be loaded
print("\nTesting model loading from wrapper...")
test_model = nn_wrapper._get_model()
print("✅ Model file can be loaded successfully!")

# Save the wrapper as PKL
output_file = "best_model_neural_network.pkl"
print(f"\n💾 Saving model to {output_file}...")
with open(output_file, 'wb') as f:
    pickle.dump(nn_wrapper, f)
print(f"✅ Model saved successfully to {output_file}!")

# Test the saved model
print("\n🧪 Testing saved model...")
test_tweet = "I love this product! It's amazing!"
sentiment, prob, conf = nn_wrapper.predict_sentiment(test_tweet)
print(f"Test: '{test_tweet}'")
print(f"Prediction: {sentiment} (probability: {prob:.4f}, confidence: {conf:.4f})")
print("\n✅ Model export complete! You can now use best_model_neural_network.pkl in Streamlit.")


Loading model from best_model.keras...
✅ Model loaded successfully!
Saving model weights to best_model_weights.h5...
✅ Model weights saved successfully to best_model_weights.h5!
✅ Model embedding dimension: 128

Testing model loading from wrapper...
✅ Model file can be loaded successfully!

💾 Saving model to best_model_neural_network.pkl...
✅ Model saved successfully to best_model_neural_network.pkl!

🧪 Testing saved model...
Test: 'I love this product! It's amazing!'
Prediction: Negative (probability: 0.2232, confidence: 0.7768)

✅ Model export complete! You can now use best_model_neural_network.pkl in Streamlit.


In [6]:
# Predict
def predict_sentiment(text):
    cleaned = preprocess_text(clean_text(text))
    seq = tokenizer.texts_to_sequences([cleaned])
    pad = pad_sequences(seq, maxlen=MAX_SEQUENCE_LENGTH)
    pred = model.predict(pad)[0][0]
    return ('Positive' if pred > 0.5 else 'Negative', pred)

# Example
tweet = "The food at the restaurant was bad! I had a worse time."
sentiment, conf = predict_sentiment(tweet)
print(f"\n💬 '{tweet}' -> {sentiment} ({conf:.4f})")


1/1 [==============================] - 0s 50ms/step

💬 'The food at the restaurant was bad! I had a worse time.' -> Negative (0.2063)


# Transformer

In [7]:
# Part B: Implementation of Sentiment Analysis on Twitter Data with a Transformer

import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Embedding, Dense, Dropout, Layer, LayerNormalization, Input, MultiHeadAttention, GlobalAveragePooling1D

# Download NLTK data (run this once)
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Define a function to clean the tweets
def clean_text(text):
    text = str(text).lower()  # Convert text to lowercase
    text = re.sub('\[.*?\]', '', text)  # Remove text in square brackets
    text = re.sub('https?://\S+|www\.\S+', '', text)  # Remove URLs
    text = re.sub('<.*?>+', '', text)  # Remove HTML tags
    text = re.sub('@[A-Za-z0-9_]+', '', text)  # Remove mentions
    text = re.sub('#[A-Za-z0-9_]+', '', text)  # Remove hashtags
    text = re.sub(r'[^a-z\s]', '', text)  # Remove all non-alphabetic characters
    return text

df['text'] = df['text'].apply(clean_text)

# Tokenization, Stopword Removal, and Lemmatization
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(words)

df['text'] = df['text'].apply(preprocess_text)

# Split the data into training and testing sets
X = df['text']
y = df['polarity']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Tokenize and pad the sequences
MAX_WORDS = 50000  # Number of words to keep in the vocabulary
MAX_SEQUENCE_LENGTH = 100  # Max length of a tweet

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<unk>")
tokenizer.fit_on_texts(X_train)

# Convert text to sequences of integers
X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_test_sequences = tokenizer.texts_to_sequences(X_test)

# Pad the sequences to a fixed length
X_train_padded = pad_sequences(X_train_sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
X_test_padded = pad_sequences(X_test_sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')

# Step 4: Build the Keras Model (Transformer)

# Custom layer for a Transformer Block
class TransformerBlock(Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential(
            [Dense(ff_dim, activation='relu'),
             Dense(embed_dim)]
        )
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training=None):  # 👈 training defaults to None
        # Self-attention
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        # Feed-forward
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)


# Custom layer for Positional Embedding
class PositionalEmbedding(Layer):
    def __init__(self, max_len, vocab_size, embed_dim):
        super(PositionalEmbedding, self).__init__()
        self.token_emb = Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = Embedding(input_dim=max_len, output_dim=embed_dim)

    def call(self, x):
        max_len = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=max_len, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

# IMPORTANT: Using the Functional API for the Transformer model
# This allows for the residual connections required by the architecture.
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input

# Hyperparameters for the Transformer model
embed_dim = 128  # Embedding size for each token
num_heads = 2    # Number of attention heads
ff_dim = 32      # Hidden layer size in feedforward network

inputs = Input(shape=(MAX_SEQUENCE_LENGTH,))
x = PositionalEmbedding(MAX_SEQUENCE_LENGTH, MAX_WORDS, embed_dim)(inputs)
x = TransformerBlock(embed_dim, num_heads, ff_dim)(x)
x = GlobalAveragePooling1D()(x)
x = Dropout(0.5)(x)
outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs=inputs, outputs=outputs)

# Compile the model
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

# Step 5: Train the Model

BATCH_SIZE = 1024
EPOCHS = 5

print("\nStarting Transformer model training...")
history = model.fit(
    X_train_padded,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1
)

# Step 6: Evaluate the Model

loss, accuracy = model.evaluate(X_test_padded, y_test)
print(f"\nModel Evaluation on Test Data:")
print(f"Loss: {loss:.4f}")
print(f"Accuracy: {accuracy:.4f}")



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\affan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\affan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\affan\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 100)]             0         
                                                                 
 positional_embedding (Posit  (None, 100, 128)         6412800   
 ionalEmbedding)                                                 
                                                                 
 transformer_block (Transfor  (None, 100, 128)         140832    
 merBlock)                                                       
                                                                 
 global_average_pooling1d (G  (None, 128)              0         
 lobalAveragePooling1D)                                          
                                                                 
 dropout_6 (Dropout)         (None, 128)               0         
                                                             

In [8]:
# Example of a negative prediction
def predict_sentiment(text):
    cleaned_text = preprocess_text(clean_text(text))
    sequence = tokenizer.texts_to_sequences([cleaned_text])
    padded_sequence = pad_sequences(sequence, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
    prediction = model.predict(padded_sequence)[0][0]
    sentiment = 'Positive' if prediction > 0.5 else 'Negative'
    return sentiment, prediction

# Test with a new tweet
test_tweet = "very bad  horrible food! I had worse time."
sentiment, confidence = predict_sentiment(test_tweet)
print(f"\nTest Tweet: '{test_tweet}'")
print(f"Predicted Sentiment: {sentiment} (Confidence: {confidence:.4f})")


1/1 [==============================] - 0s 384ms/step

Test Tweet: 'very bad  horrible food! I had worse time.'
Predicted Sentiment: Negative (Confidence: 0.0181)


In [9]:
# Example of a positive prediction
def predict_sentiment(text):
    cleaned_text = preprocess_text(clean_text(text))
    sequence = tokenizer.texts_to_sequences([cleaned_text])
    padded_sequence = pad_sequences(sequence, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
    prediction = model.predict(padded_sequence)[0][0]
    sentiment = 'Positive' if prediction > 0.5 else 'Negative'
    return sentiment, prediction

# Test with a new tweet
test_tweet = "The food at the restaurant was amazing! I had a great time."
sentiment, confidence = predict_sentiment(test_tweet)
print(f"\nTest Tweet: '{test_tweet}'")
print(f"Predicted Sentiment: {sentiment} (Confidence: {confidence:.4f})")


1/1 [==============================] - 0s 24ms/step

Test Tweet: 'The food at the restaurant was amazing! I had a great time.'
Predicted Sentiment: Positive (Confidence: 0.9880)
